# Customer Support Auto-Resolution Model: Fine-Tuning Guide (Colab)

This notebook provides a complete, end-to-end workflow for fine-tuning Mistral-7B-Instruct-v0.2 to handle customer support inquiries with structured responses.

## Step 1: Install Dependencies

We will install the core ML stack optimized for QLoRA.

In [ ]:
!pip install -q -U transformers peft trl bitsandbytes datasets accelerate scipy rouge-score bert-score pyyaml

## Step 2: Setup Environment & Configuration

Use the Colab 'Keys' (secrets) panel to add your `HF_TOKEN` and `HF_USER_NAME`.

In [ ]:
import os
from google.colab import userdata

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    os.environ['HF_USER_NAME'] = userdata.get('HF_USER_NAME')
    print('Environment Variables Set!')
except:
    print('Please add HF_TOKEN and HF_USER_NAME to Colab Secrets (Left Sidebar -> Keys Icon).')

## Step 3: Data Preparation

Fetching the `bitext` dataset and formatting it into Mistral instructions.

In [ ]:
import json
from datasets import load_dataset
from sklearn.model_selection import train_test_split

def format_mistral_instruction(instruction, response, intent=None, action=None):
    structured_response = {
        "intent": intent,
        "response": response,
        "action": action or "suggest_next_step"
    }
    response_text = json.dumps(structured_response, indent=2)
    return f"<s>[INST] {instruction} [/INST] {response_text} </s>"

# Load Dataset
ds = load_dataset('bitext/Bitext-customer-support-llm-chatbot-training-dataset', split='train')
formatted_data = []

print("Formatting dataset...")
for entry in ds.select(range(min(2000, len(ds)))): # Limiting to 2000 for faster training demo
    formatted_data.append({"text": format_mistral_instruction(entry['instruction'], entry['response'], entry['intent'], f"handle_{entry['intent']}")})

# Save locally
train_ds, test_ds = train_test_split(formatted_data, test_size=0.1, random_state=42)
with open('train.jsonl', 'w') as f:
    for e in train_ds: f.write(json.dumps(e) + '')
with open('test.jsonl', 'w') as f:
    for e in test_ds: f.write(json.dumps(e) + '')
print(f"Data Ready! (Train: {len(train_ds)}, Test: {len(test_ds)})")

## Step 4: Fine-Tuning with QLoRA

Loading the model in 4-bit and attaching LoRA adapters.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_dataset

model_id = "mistralai/Mistral-7B-Instruct-v0.2"

# 4-bit Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load Tokenizer & Model
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    quantization_config=bnb_config, 
    device_map="auto"
)
model = prepare_model_for_kbit_training(model)

# LoRA Config
peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)
model = get_peft_model(model, peft_config)

# Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=load_dataset('json', data_files='train.jsonl', split='train'),
    eval_dataset=load_dataset('json', data_files='test.jsonl', split='train'),
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=1024,
    tokenizer=tokenizer,
    args=TrainingArguments(
        output_dir="./results",
        num_train_epochs=1, # 1 Epoch for demo
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        optim="paged_adamw_32bit",
        learning_rate=2.0e-4,
        logging_steps=10,
        fp16=True,
        evaluation_strategy="steps",
        eval_steps=50,
        save_strategy="epoch"
    ),
)

trainer.train()
trainer.model.save_pretrained("./mistral-support-adapter")
print("Training Complete!")

## Step 5: Model Evaluation

Testing the model on a few samples from the test set.

In [ ]:
# Sample inference
test_query = "I received my order #12345 but it is damaged. What should I do?"
prompt = f"<s>[INST] {test_query} [/INST]"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.1, do_sample=True)

print(tokenizer.decode(outputs[0], skip_special_tokens=True).split("[/INST]")[-1].strip())